# T2.3 — YOLO11n detector (5 classes) + YOLOv8n comparison [COLAB]

Trains **YOLO11n (COCO-pretrained)** on the leak-free detection re-split, with **sonar-safe
augmentation only** (flip along-track, mild brightness/scale — no across-track flip, no hue,
no mosaic). Also runs **YOLOv8n** on the same recipe so the version choice is backed by our own
numbers. Exports YOLO11n to ONNX (≤15 MB target), registers it, and reports per-class AP with
**REAL vs SYNTHETIC separated (R4)** + a per-source table (gives the SCTD mAP50 the spec asks for).

**Before running:** put the full **`Samudra_Prahari_Dataset_v2/`** in your Google Drive (YOLO
needs the whole train split — the T2.1 zip only has the test crops). Run top-to-bottom on GPU.

In [ ]:
# 0. Setup
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!git clone https://github.com/ASugandhan/SamudraPrahari.git || (cd SamudraPrahari && git pull)
%cd SamudraPrahari
!pip install -q -e '.[models]' ultralytics

In [ ]:
# 1. Data — needs the FULL Samudra_Prahari_Dataset_v2 (for the train split), then materialise re-split
import os, glob, subprocess
from pathlib import Path
DRIVE = '/content/drive/MyDrive'      # <-- where the dataset is in your Drive

if not Path('_resplit/detection/data.yaml').exists():
    if not Path('Samudra_Prahari_Dataset_v2').exists():
        for c in glob.glob(f'{DRIVE}/**/Samudra_Prahari_Dataset_v2', recursive=True):
            os.symlink(c, 'Samudra_Prahari_Dataset_v2'); break
    assert Path('Samudra_Prahari_Dataset_v2').exists(), (
        f'Put Samudra_Prahari_Dataset_v2/ in your Drive ({DRIVE}). '
        'YOLO needs the full train split (the T2.1 zip has only test crops).')
    subprocess.run(['python', 'scripts/data/index_samudra_v2.py'], check=True)
    subprocess.run(['python', 'scripts/data/materialize_resplit.py'], check=True)
assert Path('_resplit/detection/data.yaml').exists()
print('detection ready — train imgs:', len(glob.glob('_resplit/detection/images/train/*.jpg')))

In [ ]:
# 2. Config (sonar-safe augmentation lives in configs/training.yaml)
import yaml
CFG = yaml.safe_load(open('configs/training.yaml'))['detector']
AUG = dict(CFG['augment'])
print('model', CFG['model'], '| compare', CFG['compare_with'], '| epochs', CFG['epochs'])
print('sonar-safe aug:', AUG)

In [ ]:
# 3. Train YOLO11n (COCO-pretrained) — sonar-safe aug only
from ultralytics import YOLO
y11 = YOLO(CFG['model'])                      # downloads COCO-pretrained yolo11n.pt
y11.train(data=CFG['data'], imgsz=CFG['imgsz'], epochs=CFG['epochs'], name='yolo11n',
          pretrained=True, **AUG)
m11 = y11.val(data=CFG['data'], split='test', verbose=False)
print(f'YOLO11n TEST: mAP50 {float(m11.box.map50):.3f}  mAP50-95 {float(m11.box.map):.3f}')

In [ ]:
# 4. Train YOLOv8n on the SAME recipe (comparison — justify the version with our own numbers)
y8 = YOLO(CFG['compare_with'])                # COCO-pretrained yolov8n.pt
y8.train(data=CFG['data'], imgsz=CFG['imgsz'], epochs=CFG['epochs'], name='yolov8n',
         pretrained=True, **AUG)
m8 = y8.val(data=CFG['data'], split='test', verbose=False)
print(f'YOLOv8n TEST: mAP50 {float(m8.box.map50):.3f}  mAP50-95 {float(m8.box.map):.3f}')
print(f'COMPARISON mAP50  ->  YOLO11n {float(m11.box.map50):.3f}  vs  YOLOv8n {float(m8.box.map50):.3f}')

In [ ]:
# 5. Export YOLO11n -> ONNX (<=15 MB target) + register
import shutil, json, os
y11.export(format='onnx', opset=13, dynamic=True)                 # writes best.onnx next to best.pt
onnx_src = 'runs/detect/yolo11n/weights/best.onnx'
shutil.copy(onnx_src, 'models/stage_b_yolo11n.onnx')
print('ONNX size:', round(os.path.getsize('models/stage_b_yolo11n.onnx')/1e6, 2), 'MB (target <=15)')
Path('out').mkdir(exist_ok=True)
json.dump({'task': 'T2.3',
           'yolo11n_test_mAP50': round(float(m11.box.map50), 4),
           'yolo11n_test_mAP50_95': round(float(m11.box.map), 4),
           'yolov8n_test_mAP50': round(float(m8.box.map50), 4)},
          open('out/metrics_t23.json', 'w'), indent=2)
!python scripts/register_model.py --onnx models/stage_b_yolo11n.onnx --name yolo11n --trained-on 'Detection re-split (real+synth, R4)' --metrics out/metrics_t23.json

In [ ]:
# 6. Per-class + per-source metrics (R4 real-vs-synth, R5 per-source -> SCTD mAP50)
!python scripts/eval/yolo_eval.py --split test --weights runs/detect/yolo11n/weights/best.pt --per-source

In [ ]:
# 7. Copy artifacts back to Drive
!cp models/stage_b_yolo11n.onnx models/registry.json out/metrics_t23.json /content/drive/MyDrive/ 2>/dev/null
!cp runs/detect/yolo11n/weights/best.pt /content/drive/MyDrive/yolo11n_best.pt 2>/dev/null
print('copied ONNX + best.pt + registry + metrics to Drive')

## After it runs
1. Note **YOLO11n mAP50** (target ≥ 0.85; SCTD-specific mAP50 is in the per-source table) and the
   **YOLO11n vs YOLOv8n** line.
2. Copy back into the repo: `models/stage_b_yolo11n.onnx`, `models/registry.json`,
   `out/metrics_t23.json`, and `yolo11n_best.pt` (for local `yolo_eval.py`). Commit + push.
3. Locally (optional): `pip install ultralytics` then
   `python scripts/eval/yolo_eval.py --split test --weights yolo11n_best.pt --per-source`.

**Honesty (R4):** shipwreck/aircraft are REAL — that's the headline. pipe/cylinder/ghost_net are
SYNTHETIC; the eval prints them separately and never blends them into the real headline.